# Cell 1 - Guardrails
# OBAD L1 Candidate C - A/B/C Evaluation Only

Package and Candidate C artifacts are already complete. This notebook is read-only: no SQL, no L2, no package preparation, no model fitting, no production write, and no automatic promotion. `VALID` is used for the decision gate; `TEST` is final reporting only.

In [1]:
# Cell 2 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 3 - Define independent paths
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/OBAD')
PACKAGE = (
    PROJECT_ROOT
    / 'data/dataModel/l1_adaptation/l1_candidate_c_current'
)
CANDIDATE_ARTIFACT = (
    PROJECT_ROOT
    / 'modeling/l1_tcn/artifacts_candidates'
    / 'l1_candidate_c_current/current_only'
)
ADAPTATION = (
    PROJECT_ROOT
    / 'data/realtime_audit/l1_adaptation_eval_20260715_162915'
)
AUDIT_ROOT = PROJECT_ROOT / 'data/realtime_audit'

for path in (PROJECT_ROOT, PACKAGE, CANDIDATE_ARTIFACT, ADAPTATION, AUDIT_ROOT):
    assert path.exists(), f'Missing required path: {path}'
print({'PROJECT_ROOT': PROJECT_ROOT, 'PACKAGE': PACKAGE, 'CANDIDATE_ARTIFACT': CANDIDATE_ARTIFACT, 'ADAPTATION': ADAPTATION})

{'PROJECT_ROOT': PosixPath('/content/drive/MyDrive/OBAD'), 'PACKAGE': PosixPath('/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current'), 'CANDIDATE_ARTIFACT': PosixPath('/content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts_candidates/l1_candidate_c_current/current_only'), 'ADAPTATION': PosixPath('/content/drive/MyDrive/OBAD/data/realtime_audit/l1_adaptation_eval_20260715_162915')}


In [3]:
# Cell 4 - Environment check; install only after an import failure
import importlib
import os
import platform
import subprocess
import sys

try:
    import pandas as pd
    import torch
    import psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements2.txt'), 'pyarrow', 'pyyaml', 'psutil'], check=True)
    import pandas as pd
    import torch
    import psutil

print({'python': platform.python_version(), 'pandas': pd.__version__, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'ram_gb': round(psutil.virtual_memory().total / 2**30, 2)})

{'python': '3.12.13', 'pandas': '2.2.2', 'torch': '2.11.0+cpu', 'cuda': False, 'gpu': None, 'ram_gb': 12.67}


In [ ]:
# Cell 5 - Verify evaluator source and CLI synchronization
import hashlib
import subprocess

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

EVALUATOR_FILES = [
    PROJECT_ROOT / 'inference/online/l1_candidate_evaluation.py',
    PROJECT_ROOT / 'inference/online/score_new_events.py',
    PROJECT_ROOT / 'modeling/l1_tcn/scripts/run_candidate_c_colab.py',
]
for path in EVALUATOR_FILES:
    assert path.exists(), path
    print({'path': str(path), 'modified_utc': path.stat().st_mtime, 'sha256': sha256(path)})

help_result = subprocess.run([sys.executable, 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', '--help'], cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
print(help_result.stdout)
for token in ('evaluate', '--candidate-package-dir', '--candidate-artifact-dir', '--adaptation-audit-dir'):
    assert token in help_result.stdout, f'Missing CLI contract token: {token}'

{'path': '/content/drive/MyDrive/OBAD/inference/online/l1_candidate_evaluation.py', 'modified_utc': 1784188823.0, 'sha256': '8b495abb2e012141f9cd7fc821b2befde5d783daf7ba03c55c58fe802b9e69b4'}
{'path': '/content/drive/MyDrive/OBAD/inference/online/score_new_events.py', 'modified_utc': 1784185713.0, 'sha256': 'ef88dc865c859c0ab0338ee55c0e64b270b3dcec4c4da5d33f6edbd6e037947d'}
{'path': '/content/drive/MyDrive/OBAD/modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'modified_utc': 1784173183.0, 'sha256': '98beab6e3d9e8d6879817421a5b4e1d3c8954ac4f2755285429c49ffaa6ac419'}
usage: run_candidate_c_colab.py [-h] [--package-dir CANDIDATE_PACKAGE_DIR]
                                [--candidate-package-dir CANDIDATE_PACKAGE_DIR]
                                [--source-snapshot-dir SOURCE_SNAPSHOT_DIR]
                                [--source-mode {snapshot}]
                                [--candidate-run-id CANDIDATE_RUN_ID]
                                [--adaptation-audit-dir ADAPTATION

In [ ]:
# Cell 6 - Read-only preflight
import json

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required = {
    'package_valid': PACKAGE / 'evaluation/valid_all',
    'package_test': PACKAGE / 'evaluation/test_all',
    'candidate_lenient_model': CANDIDATE_ARTIFACT / 'lenient/model_best.pt',
    'candidate_lenient_preprocessor': CANDIDATE_ARTIFACT / 'lenient/preprocessor.json',
    'candidate_lenient_thresholds': CANDIDATE_ARTIFACT / 'lenient/thresholds.json',
    'candidate_strict_model': CANDIDATE_ARTIFACT / 'strict/model_best.pt',
    'candidate_strict_preprocessor': CANDIDATE_ARTIFACT / 'strict/preprocessor.json',
    'candidate_strict_thresholds': CANDIDATE_ARTIFACT / 'strict/thresholds.json',
    'production_lenient_model': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/lenient/model_best.pt',
    'production_lenient_preprocessor': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/lenient/preprocessor.json',
    'production_lenient_thresholds': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/lenient/thresholds.json',
    'production_strict_model': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/strict/model_best.pt',
    'production_strict_preprocessor': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/strict/preprocessor.json',
    'production_strict_thresholds': PROJECT_ROOT / 'modeling/l1_tcn/artifacts/strict/thresholds.json',
}
preflight = [{'input': name, 'path': str(path), 'result': 'PASS' if path.exists() else 'FAIL'} for name, path in required.items()]
b_threshold_candidates = [ADAPTATION / '14_candidate_b_grid_thresholds.json', ADAPTATION / '16_candidate_b_thresholds.json']
preflight.append({'input': 'candidate_b_thresholds', 'path': ', '.join(map(str, b_threshold_candidates)), 'result': 'PASS' if any(path.exists() for path in b_threshold_candidates) else 'FAIL'})
print(pd.DataFrame(preflight))
assert all(item['result'] == 'PASS' for item in preflight), 'EVALUATION_BLOCKED_PREFLIGHT_FAILED'
PRODUCTION_HASHES = {str(path): file_hash(path) for name, path in required.items() if name.startswith('production_')}

                              input  \
0                     package_valid   
1                      package_test   
2           candidate_lenient_model   
3    candidate_lenient_preprocessor   
4      candidate_lenient_thresholds   
5            candidate_strict_model   
6     candidate_strict_preprocessor   
7       candidate_strict_thresholds   
8          production_lenient_model   
9   production_lenient_preprocessor   
10    production_lenient_thresholds   
11          production_strict_model   
12   production_strict_preprocessor   
13     production_strict_thresholds   
14           candidate_b_thresholds   

                                                 path result  
0   /content/drive/MyDrive/OBAD/data/dataModel/l1_...   PASS  
1   /content/drive/MyDrive/OBAD/data/dataModel/l1_...   PASS  
2   /content/drive/MyDrive/OBAD/modeling/l1_tcn/ar...   PASS  
3   /content/drive/MyDrive/OBAD/modeling/l1_tcn/ar...   PASS  
4   /content/drive/MyDrive/OBAD/modeling/l1_tcn/ar...   PASS

In [ ]:
# Cell 7 - Run the read-only A/B/C evaluator once
import datetime
import time

EVALUATION_START = time.time()
env = os.environ.copy()
env.update({'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8', 'PYTHONFAULTHANDLER': '1'})
command = [sys.executable, '-u', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'evaluate', '--candidate-package-dir', str(PACKAGE), '--candidate-artifact-dir', str(CANDIDATE_ARTIFACT), '--adaptation-audit-dir', str(ADAPTATION)]
print({'command': command, 'started_at': datetime.datetime.now().isoformat(timespec='seconds')})
result = subprocess.run(command, cwd=PROJECT_ROOT, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
EVALUATION_END = time.time()
print({'return_code': result.returncode, 'ended_at': datetime.datetime.now().isoformat(timespec='seconds'), 'elapsed_seconds': round(EVALUATION_END - EVALUATION_START, 2)})
print('=== STDOUT ===\n' + (result.stdout or '<EMPTY>'))
print('=== STDERR ===\n' + (result.stderr or '<EMPTY>'))
if result.returncode != 0:
    print('EVALUATION_FAILED')
    raise RuntimeError(f'EVALUATION_FAILED return_code={result.returncode}')
print('EVALUATION_COMPLETED')

{'command': ['/usr/bin/python3', '-u', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'evaluate', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--candidate-artifact-dir', '/content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts_candidates/l1_candidate_c_current/current_only', '--adaptation-audit-dir', '/content/drive/MyDrive/OBAD/data/realtime_audit/l1_adaptation_eval_20260715_162915'], 'started_at': '2026-07-16T08:41:57'}
{'return_code': 0, 'ended_at': '2026-07-16T09:05:48', 'elapsed_seconds': 1431.2}
=== STDOUT ===
candidate_c_runner: {"action": "evaluate", "command": ["/usr/bin/python3", "-m", "inference.online.score_new_events", "--config", "inference/online/config.example.yaml", "--evaluate-l1-retrain-candidate", "--candidate-package-dir", "data/dataModel/l1_adaptation/l1_candidate_c_current", "--candidate-artifact-dir", "modeling/l1_tcn/artifacts_candidates/l1_candidate_c_current/current_only", "--adaptation-audi

In [ ]:
# Cell 8 - Locate only the audit created by Cell 7
evaluation_dirs = sorted(AUDIT_ROOT.glob('l1_candidate_c_eval_*'), key=lambda path: path.stat().st_mtime, reverse=True)
valid_dirs = []
for path in evaluation_dirs:
    summary_path = path / 'summary.json'
    if not summary_path.exists() or path.stat().st_mtime < EVALUATION_START - 5 or path.stat().st_mtime > EVALUATION_END + 120:
        continue
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        if summary.get('result') == 'PASS':
            valid_dirs.append(path)
    except Exception:
        pass
assert valid_dirs, 'No successful evaluation audit was created during Cell 7'
LATEST_EVAL = valid_dirs[0]
print('LATEST_EVAL =', LATEST_EVAL)
for path in sorted(LATEST_EVAL.rglob('*')):
    if path.is_file():
        print(path.relative_to(LATEST_EVAL), path.stat().st_size)

LATEST_EVAL = /content/drive/MyDrive/OBAD/data/realtime_audit/l1_candidate_c_eval_20260716_084204
candidate_abc_comparison.json 8601
candidate_abc_scores.parquet 69347078
candidate_c_decision_gate.json 202
candidate_c_historical_regression.json 84
candidate_c_metrics_by_machine.json 127872
candidate_c_metrics_global.json 7497
candidate_c_strict_lenient_overlap.json 49170
summary.json 323


In [ ]:
# Cell 9 - Validate the evaluation output contract
REPORTS = {
    'summary': 'summary.json',
    'candidate_c_global_metrics': 'candidate_c_metrics_global.json',
    'candidate_c_metrics_by_machine': 'candidate_c_metrics_by_machine.json',
    'strict_lenient_overlap': 'candidate_c_strict_lenient_overlap.json',
    'abc_comparison': 'candidate_abc_comparison.json',
    'decision_gate': 'candidate_c_decision_gate.json',
    'historical_regression': 'candidate_c_historical_regression.json',
    'technical_validation': 'summary.json',
}
rows = []
for report, relative in REPORTS.items():
    path = LATEST_EVAL / relative
    status = None
    if path.exists() and path.suffix == '.json':
        status = json.loads(path.read_text(encoding='utf-8')).get('result')
    rows.append({'report': report, 'exists': path.exists(), 'size': path.stat().st_size if path.exists() else 0, 'result_or_status': status})
contract = pd.DataFrame(rows)
print(contract)
assert contract['exists'].all(), 'EVALUATION_OUTPUT_CONTRACT_FAILED'

                           report  exists    size result_or_status
0                         summary    True     323             PASS
1      candidate_c_global_metrics    True    7497             None
2  candidate_c_metrics_by_machine    True  127872             None
3          strict_lenient_overlap    True   49170             None
4                  abc_comparison    True    8601             None
5                   decision_gate    True     202             None
6           historical_regression    True      84    NOT_AVAILABLE
7            technical_validation    True     323             PASS


In [ ]:
# Cell 10 - Display the primary reports
for title, relative in [
    ('1. Summary', 'summary.json'),
    ('2. Decision gate', 'candidate_c_decision_gate.json'),
    ('3. A/B/C comparison', 'candidate_abc_comparison.json'),
    ('4. Candidate C global metrics', 'candidate_c_metrics_global.json'),
    ('5. Per-machine metrics', 'candidate_c_metrics_by_machine.json'),
    ('6. Strict/lenient overlap', 'candidate_c_strict_lenient_overlap.json'),
    ('7. Historical regression', 'candidate_c_historical_regression.json'),
]:
    print('\n===== ' + title + ' =====')
    payload = json.loads((LATEST_EVAL / relative).read_text(encoding='utf-8'))
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    if relative == 'candidate_c_metrics_by_machine.json':
        for machine_id in ('49', '51', '58'):
            print(f'\nMachine {machine_id}:', json.dumps(payload.get('priority_machines', {}).get(machine_id, 'NOT_PRESENT'), ensure_ascii=False, indent=2))


===== 1. Summary =====
{
  "result": "PASS",
  "decision": "KEEP_CURRENT_MODEL_AND_THRESHOLDS",
  "output_dir": "/content/drive/MyDrive/OBAD/data/realtime_audit/l1_candidate_c_eval_20260716_084204",
  "write_sql_enabled": false,
  "l2_prediction_run": false,
  "production_artifacts_overwritten": false,
  "production_checkpoint_updated": false
}

===== 2. Decision gate =====
{
  "decision": "KEEP_CURRENT_MODEL_AND_THRESHOLDS",
  "selection_split": "VALID",
  "test_used_for_selection": false,
  "historical_regression_result": "NOT_AVAILABLE",
  "automatic_promotion": false
}

===== 3. A/B/C comparison =====
{
  "exact_window_identity": "PASS",
  "by_split": {
    "TEST": {
      "exact_window_count": 170930,
      "unique_event_count": 170930,
      "candidates": {
        "candidate_a": {
          "total_support": 170930,
          "scored_window_support": 170930,
          "not_scored_window_support": 0,
          "normal_lenient_support": 169270,
          "normal_false_positive_rat

In [ ]:
# Cell 11 - Independent read-only sanity checks
scores = pd.read_parquet(LATEST_EVAL / 'candidate_abc_scores.parquet')
assert not scores.duplicated(['machine_id', 'event_id']).any()
assert not any(column.endswith(('_x', '_y')) for column in scores.columns)
for split_name in ('VALID', 'TEST'):
    split_keys = scores.loc[scores['split_name'] == split_name, ['machine_id', 'event_id']]
    assert not split_keys.empty, f'No {split_name} windows'
    assert not split_keys.duplicated().any(), f'Duplicate {split_name} target key'
    ready = scores.loc[(scores['split_name'] == split_name) & (scores['window_ready_flag'] == 1)]
    assert ready[['candidate_a_score_lenient', 'candidate_b_score_lenient', 'candidate_c_score_lenient']].notna().all().all()
assert (scores['candidate_a_score_lenient'].fillna(-1) == scores['candidate_b_score_lenient'].fillna(-1)).all()
assert (scores['candidate_a_score_strict'].fillna(-1) == scores['candidate_b_score_strict'].fillna(-1)).all()
assert {'candidate_a_is_behavior_anomaly', 'candidate_b_is_behavior_anomaly', 'candidate_c_is_behavior_anomaly'}.issubset(scores.columns)
leakage = json.loads((PACKAGE / 'manifests/split_leakage_report.json').read_text(encoding='utf-8'))
assert leakage.get('window_cross_machine_count') == 0
assert leakage.get('window_cross_segment_count') == 0
assert leakage.get('window_cross_split_count') == 0
assert all(file_hash(Path(path)) == value for path, value in PRODUCTION_HASHES.items())
sanity = {'result': 'PASS', 'same_exact_valid_keys': True, 'same_exact_test_keys': True, 'candidate_b_uses_candidate_a_scores': True, 'no_duplicate_machine_event_key': True, 'no_unexpected_suffix_columns': True, 'no_window_cross_machine': True, 'no_window_cross_segment': True, 'no_window_cross_split': True, 'production_artifacts_unchanged': True}
(LATEST_EVAL / 'candidate_abc_sanity_report.json').write_text(json.dumps(sanity, indent=2), encoding='utf-8')
print(json.dumps(sanity, indent=2))

{
  "result": "PASS",
  "same_exact_valid_keys": true,
  "same_exact_test_keys": true,
  "candidate_b_uses_candidate_a_scores": true,
  "no_duplicate_machine_event_key": true,
  "no_unexpected_suffix_columns": true,
  "no_window_cross_machine": true,
  "no_window_cross_segment": true,
  "no_window_cross_split": true,
  "production_artifacts_unchanged": true
}


In [ ]:
# Cell 12 - Package only the completed evaluation report
import shutil
zip_base = Path('/content/l1_candidate_c_eval_latest')
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=LATEST_EVAL.parent, base_dir=LATEST_EVAL.name))
print({'zip_path': str(zip_path), 'size_bytes': zip_path.stat().st_size})

{'zip_path': '/content/l1_candidate_c_eval_latest.zip', 'size_bytes': 60072848}


In [ ]:
# Cell 13 - OPTIONAL diagnostics for a failed evaluation only
# Run this cell only after Cell 7 fails. It does not re-run evaluation.
from shutil import disk_usage

print('OPTIONAL_DIAGNOSTICS_ONLY')
print('project_root=', PROJECT_ROOT)
print('package_exists=', PACKAGE.exists())
print('candidate_artifact_exists=', CANDIDATE_ARTIFACT.exists())
print('adaptation_exists=', ADAPTATION.exists())
print('drive_free_bytes=', disk_usage('/content/drive').free)
recent = sorted(AUDIT_ROOT.glob('l1_candidate_c_eval_*'), key=lambda path: path.stat().st_mtime, reverse=True)[:10]
for path in recent:
    print(path, 'summary_exists=', (path / 'summary.json').exists())
for path in PROJECT_ROOT.rglob('*.log'):
    if path.stat().st_mtime >= EVALUATION_START:
        print('recent_log=', path)


In [4]:
#cell đánh giá lựa chọn A sau khi test 3 case a/B/c
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

PROJECT_ROOT = Path("/content/drive/MyDrive/OBAD")

DRIVE_EVAL = (
    PROJECT_ROOT
    / "data/realtime_audit/l1_candidate_c_eval_20260716_084204"
)

DRIVE_PACKAGE = (
    PROJECT_ROOT
    / "data/dataModel/l1_adaptation/l1_candidate_c_current"
)

DRIVE_ADAPTATION = (
    PROJECT_ROOT
    / "data/realtime_audit/l1_adaptation_eval_20260715_162915"
)

LOCAL_ROOT = Path("/content/obad_post_eval")
LOCAL_EVAL = LOCAL_ROOT / "evaluation"
LOCAL_PACKAGE = LOCAL_ROOT / "candidate_package"
LOCAL_ADAPTATION = LOCAL_ROOT / "adaptation"
LOCAL_OUTPUT = LOCAL_ROOT / "output"

for path in [
    LOCAL_EVAL,
    LOCAL_PACKAGE,
    LOCAL_ADAPTATION,
    LOCAL_OUTPUT,
]:
    path.mkdir(parents=True, exist_ok=True)

def rsync_directory(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "rsync",
            "-a",
            "--info=progress2",
            f"{source}/",
            f"{destination}/",
        ],
        check=True,
    )

print("=== COPY EVALUATION AUDIT TO LOCAL SSD ===")
rsync_directory(DRIVE_EVAL, LOCAL_EVAL)

print("=== COPY CANONICAL PARTITIONS TO LOCAL SSD ===")
rsync_directory(
    DRIVE_PACKAGE / "canonical",
    LOCAL_PACKAGE / "canonical",
)

if (DRIVE_PACKAGE / "manifests").exists():
    rsync_directory(
        DRIVE_PACKAGE / "manifests",
        LOCAL_PACKAGE / "manifests",
    )

print("=== COPY ADAPTATION AUDIT TO LOCAL SSD ===")
rsync_directory(DRIVE_ADAPTATION, LOCAL_ADAPTATION)

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["PYTHONIOENCODING"] = "utf-8"
environment["PYTHONFAULTHANDLER"] = "1"

command = [
    sys.executable,
    "-u",
    "-m",
    "inference.online.l1_candidate_post_evaluation",
    "--evaluation-audit-dir",
    str(LOCAL_EVAL),
    "--candidate-package-dir",
    str(LOCAL_PACKAGE),
    "--adaptation-audit-dir",
    str(LOCAL_ADAPTATION),
    "--output-root",
    str(LOCAL_OUTPUT),
]

print("=== RUN LOCAL POST-EVALUATION AUDIT ===")
print(" ".join(command), flush=True)

started = time.time()

result = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    check=False,
)

print({
    "return_code": result.returncode,
    "elapsed_seconds": round(time.time() - started, 2),
})

assert result.returncode == 0, "POST_EVALUATION_AUDIT_FAILED"

runs = sorted(
    LOCAL_OUTPUT.glob("l1_candidate_c_post_eval_*"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

assert runs, "Không tìm thấy post-evaluation output"

LATEST_POST_EVAL = runs[0]

print("LATEST_POST_EVAL =", LATEST_POST_EVAL)

drive_destination = PROJECT_ROOT / "data/realtime_audit"

subprocess.run(
    [
        "rsync",
        "-a",
        "--info=progress2",
        str(LATEST_POST_EVAL),
        f"{drive_destination}/",
    ],
    check=True,
)

print("POST-EVALUATION AUDIT COMPLETED")

from pathlib import Path
import json
import shutil

PROJECT_ROOT = Path("/content/drive/MyDrive/OBAD")

POST_EVAL = (
    PROJECT_ROOT
    / "data/realtime_audit"
    / "l1_candidate_c_post_eval_20260717_013253"
)

assert POST_EVAL.exists(), POST_EVAL

print("=== POST-EVALUATION FILES ===")

for path in sorted(POST_EVAL.rglob("*")):
    if path.is_file():
        print({
            "name": str(path.relative_to(POST_EVAL)),
            "size": path.stat().st_size,
        })

required_reports = [
    "00_summary.json",
    "candidate_ac_disagreement_global.json",
    "candidate_ac_disagreement_by_machine.json",
    "future_label_contract_audit.json",
    "future_label_prevalence_by_machine.json",
    "future_label_lead_time_distribution.json",
    "candidate_c_historical_regression_recovered.json",
    "candidate_final_decision_rationale.json",
]

print("\n=== REQUIRED REPORT CHECK ===")

for name in required_reports:
    path = POST_EVAL / name
    print(path.exists(), name)

missing = [
    name
    for name in required_reports
    if not (POST_EVAL / name).exists()
]

print("\nMISSING:", missing)

# Tạo thư mục tạm chỉ chứa JSON nhỏ cần gửi
EXPORT_DIR = Path("/content/l1_candidate_c_post_eval_reports")

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(parents=True)

for name in required_reports:
    source = POST_EVAL / name

    if source.exists():
        shutil.copy2(source, EXPORT_DIR / name)

zip_path = shutil.make_archive(
    "/content/l1_candidate_c_post_eval_reports",
    "zip",
    root_dir=EXPORT_DIR,
)

print("\nZIP:", zip_path)

=== COPY EVALUATION AUDIT TO LOCAL SSD ===
=== COPY CANONICAL PARTITIONS TO LOCAL SSD ===
=== COPY ADAPTATION AUDIT TO LOCAL SSD ===
=== RUN LOCAL POST-EVALUATION AUDIT ===
/usr/bin/python3 -u -m inference.online.l1_candidate_post_evaluation --evaluation-audit-dir /content/obad_post_eval/evaluation --candidate-package-dir /content/obad_post_eval/candidate_package --adaptation-audit-dir /content/obad_post_eval/adaptation --output-root /content/obad_post_eval/output
{'return_code': 0, 'elapsed_seconds': 80.28}
LATEST_POST_EVAL = /content/obad_post_eval/output/l1_candidate_c_post_eval_20260717_013253
POST-EVALUATION AUDIT COMPLETED
